In [1]:
# ============================================================
# 1. Imports
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Optional models
try:
    from xgboost import XGBRegressor
    xgb_available = True
except ImportError:
    xgb_available = False

try:
    from catboost import CatBoostRegressor
    catboost_available = True
except ImportError:
    catboost_available = False


In [12]:
# ============================================================
# 2. Load Data
# ============================================================

data_path = "gem_final_combined_data_model.csv"
df = pd.read_csv(data_path)

df.head()


,g_id,address_count,total_pop,percent_60to74,percent_75andUp,percent_households_no_kids,percent_1floor,percent_2floor,percent_3to5floor,percent_own_occ,percent_priv_one,percent_res_one_dwell,income,price_sq_m
0,10101,3,16118,18.31,9.91,41.24,41.39,44.38,13.74,15.59,42.96,67.64,32774,312.70
1,10201,1,1933,21.73,15.52,43.83,71.69,24.71,3.60,4.20,35.91,79.73,24956,245.10
2,10301,4,1943,23.83,16.62,52.42,66.16,31.52,2.32,9.46,41.13,80.54,30022,326.80
3,10302,1,1881,24.46,10.21,46.79,70.85,27.43,1.72,4.45,38.52,80.97,27442,239.20
4,10303,0,2078,22.18,8.57,46.07,54.01,42.48,3.51,3.44,31.98,79.80,30000,278.20


In [13]:
# ============================================================
# 3. Basic Inspection
# ============================================================

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2112 entries, 0 to 2111
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   g_id                        2112 non-null   int64  
 1   address_count               2112 non-null   int64  
 2   total_pop                   2112 non-null   int64  
 3   percent_60to74              2112 non-null   float64
 4   percent_75andUp             2112 non-null   float64
 5   percent_households_no_kids  2112 non-null   float64
 6   percent_1floor              2112 non-null   float64
 7   percent_2floor              2112 non-null   float64
 8   percent_3to5floor           2112 non-null   float64
 9   percent_own_occ             2112 non-null   float64
 10  percent_priv_one            2112 non-null   float64
 11  percent_res_one_dwell       2112 non-null   float64
 12  income                      2112 non-null   int64  
 13  price_sq_m                  2112 

In [4]:
df.describe()


,g_id,address_count,total_pop,pop_60to74,percent_60to74,pop_75andUp,percent_75andUp,households_no_kids,percent_households_no_kids,percent_1floor,...,abs_2floor,percent_3to5floor,abs_3to5floor,percent_own_occ,abs_own_occ,percent_priv_one,abs_priv_one,percent_res_one_dwell,abs_res_one_dwell,income
count,2112.000000,2112.000000,2112.000000,2112.000000,2112.000000,2112.000000,2112.000000,2112.000000,2112.000000,2112.000000,...,2112.000000,2112.000000,2112.000000,2112.000000,2112.000000,2112.000000,2112.000000,2112.000000,2112.000000,2112.000000
mean,44186.855587,1.146780,4348.134943,770.623580,19.769242,425.786458,10.186098,497.479640,41.346747,41.051548,...,517.910511,10.664545,159.856534,9.016591,262.972538,32.002372,757.434659,67.195473,736.244318,24788.359848
std,19719.752164,3.313731,14859.579847,2205.043617,2.911213,1317.213380,2.366813,1563.219436,4.795034,20.317787,...,811.793804,9.750993,538.026467,7.884885,1195.268326,5.980364,3252.405706,13.795487,1120.356495,3246.777284
min,10101.000000,0.000000,43.000000,4.000000,9.300000,5.000000,2.590000,3.000000,24.140000,2.200000,...,6.000000,0.000000,0.000000,0.000000,0.000000,14.040000,6.000000,0.630000,6.000000,10153.000000
25%,31227.750000,0.000000,1155.750000,232.000000,17.810000,110.000000,8.540000,135.750000,38.475000,23.595000,...,198.750000,4.117500,24.000000,2.830000,15.000000,27.905000,142.000000,59.367500,306.000000,22514.750000
50%,41016.500000,0.000000,1865.500000,376.500000,19.525000,187.000000,9.900000,225.000000,41.575000,37.345000,...,342.000000,7.415000,52.000000,6.640000,49.000000,31.485000,240.000000,68.185000,505.500000,24562.500000
75%,61629.250000,1.000000,3329.250000,647.000000,21.420000,341.250000,11.502500,405.000000,44.130000,57.840000,...,588.250000,14.397500,127.000000,12.925000,158.250000,35.552500,451.250000,76.747500,814.250000,26659.250000
max,92301.000000,67.000000,305314.000000,42836.000000,46.300000,27107.000000,27.640000,34329.000000,66.670000,89.730000,...,18393.000000,89.010000,11657.000000,42.670000,35419.000000,56.930000,72761.000000,96.000000,25982.000000,41355.000000


In [14]:
# ============================================================
# 4. Define Target and Features
# ============================================================

TARGET = "address_count"

X = df.drop(columns=[TARGET, "g_id"])
y = df[TARGET]

# Identify column types
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()

categorical_cols, numerical_cols


(['price_sq_m'],
 ['total_pop',
  'percent_60to74',
  'percent_75andUp',
  'percent_households_no_kids',
  'percent_1floor',
  'percent_2floor',
  'percent_3to5floor',
  'percent_own_occ',
  'percent_priv_one',
  'percent_res_one_dwell',
  'income'])

In [16]:
# ============================================================
# 5. Train / Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [17]:
# ============================================================
# 6. Preprocessing Pipeline (for sklearn models)
# ============================================================

numeric_transformer = "passthrough"

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)


In [18]:
# ============================================================
# 7. Evaluation Helper Function
# ============================================================

def evaluate_model(model, X_test, y_test):
    preds = model.predict(X_test)
    return {
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "MAE": mean_absolute_error(y_test, preds),
        "R2": r2_score(y_test, preds)
    }

In [19]:
linreg_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

linreg_pipeline.fit(X_train, y_train)

linreg_results = evaluate_model(linreg_pipeline, X_test, y_test)
linreg_results


{'RMSE': np.float64(1.4967159547761564),
 'MAE': 0.9895545195244001,
 'R2': 0.3279118385542279}

In [20]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

rf_pipeline.fit(X_train, y_train)

rf_results = evaluate_model(rf_pipeline, X_test, y_test)
rf_results


{'RMSE': np.float64(1.3922626447145712),
 'MAE': 0.831189913317573,
 'R2': 0.41844631636133156}

In [21]:
if xgb_available:
    xgb_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", XGBRegressor(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                objective="reg:squarederror"
            ))
        ]
    )

    xgb_pipeline.fit(X_train, y_train)
    xgb_results = evaluate_model(xgb_pipeline, X_test, y_test)
    xgb_results
else:
    print("XGBoost not installed.")


In [22]:
xgb_results

{'RMSE': np.float64(1.3802783396498752),
 'MAE': 0.8808932900428772,
 'R2': 0.4284150004386902}

In [24]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.6 MB/s eta 0:00:00


In [29]:
from catboost import CatBoostRegressor


In [30]:
# Identify categorical feature indices (CatBoost needs indices, not names)
cat_features = [X.columns.get_loc(col) for col in categorical_cols]
# Initialize model
cb_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)
# Fit model
cb_model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    use_best_model=True
)
# Evaluate
cb_results = evaluate_model(cb_model, X_test, y_test)
cb_results


0:	learn: 3.4977183	test: 1.7735044	best: 1.7735044 (0)	total: 64.8ms	remaining: 1m 4s
100:	learn: 1.0924457	test: 1.1610390	best: 1.1609037 (98)	total: 2.45s	remaining: 21.8s
200:	learn: 0.8057258	test: 1.1691945	best: 1.1592317 (127)	total: 4.31s	remaining: 17.1s
300:	learn: 0.6391253	test: 1.1834371	best: 1.1592317 (127)	total: 5.83s	remaining: 13.5s
400:	learn: 0.5368284	test: 1.1903626	best: 1.1592317 (127)	total: 7.27s	remaining: 10.9s
500:	learn: 0.4551499	test: 1.1947072	best: 1.1592317 (127)	total: 8.59s	remaining: 8.55s
600:	learn: 0.3935303	test: 1.1994191	best: 1.1592317 (127)	total: 9.9s	remaining: 6.57s
700:	learn: 0.3368940	test: 1.2019175	best: 1.1592317 (127)	total: 12.2s	remaining: 5.22s
800:	learn: 0.2937194	test: 1.2046819	best: 1.1592317 (127)	total: 13.4s	remaining: 3.33s
900:	learn: 0.2586332	test: 1.2068133	best: 1.1592317 (127)	total: 14.5s	remaining: 1.6s
999:	learn: 0.2259112	test: 1.2091035	best: 1.1592317 (127)	total: 15.7s	remaining: 0us

bestTest = 1.1592

{'RMSE': np.float64(1.1592317080712518),
 'MAE': 0.787242788382069,
 'R2': 0.5968302191206787}

In [31]:
feature_importance = cb_model.get_feature_importance()

fi_df = pd.DataFrame({
    "feature": X.columns,
    "importance": feature_importance
}).sort_values("importance", ascending=False)

fi_df.head(20)


,feature,importance
0,total_pop,63.217722
5,percent_2floor,10.918761
10,income,6.970915
2,percent_75andUp,4.060459
7,percent_own_occ,4.039116
4,percent_1floor,2.768863
8,percent_priv_one,2.239442
9,percent_res_one_dwell,1.766697
1,percent_60to74,1.634138
3,percent_households_no_kids,1.223606


from matplotlib import pyplot as plt
_df_0['index'].plot(kind='hist', bins=20, title='index')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_1['importance'].plot(kind='hist', bins=20, title='importance')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_2.plot(kind='scatter', x='index', y='importance', s=32, alpha=.8)
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_3['index'].plot(kind='line', figsize=(8, 4), title='index')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_4['importance'].plot(kind='line', figsize=(8, 4), title='importance')
plt.gca().spines[['top', 'right']].set_visible(False)